# Week 4 — Predictive Modeling and Optimization

**Author:** Subhajit Sangram

All paths are relative to the project root. Run from the repository root.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("data/week3_4_analysis_logistics.csv", parse_dates=["order_date"])
df = df.sort_values("order_date").reset_index(drop=True)
display(df.head())

,order_id,order_date,state,city,category,transport_mode,shipment_volume,weight_kg,distance_km,fuel_price_per_litre,weather,processing_hours,delivery_days,promised_days,late_flag,delay_days,freight_cost,transport_cost,inventory_level,stockout_flag,customer_rating
0,ORD101487,2025-01-01,Tamil Nadu,Coimbatore,Sports,Truck,8,4.63,50.1,88.02,Rain,8.3,4.95,5.0,0,0.00,188.88,252.80,127,0,4.4
1,ORD101995,2025-01-01,Karnataka,Mangaluru,Books,Bike,5,1.85,24.3,97.94,Normal,9.1,1.70,1.0,1,0.70,140.33,127.62,43,0,3.5
2,ORD102441,2025-01-01,West Bengal,Siliguri,Grocery,Bike,6,1.79,160.1,103.73,Normal,6.1,3.25,2.5,1,0.75,256.34,312.06,5,1,3.7
3,ORD103673,2025-01-01,Karnataka,Mysuru,Home & Kitchen,Van,2,2.06,156.1,90.85,Normal,6.8,3.76,4.5,0,0.00,211.52,251.81,5,1,4.2
4,ORD102788,2025-01-02,Maharashtra,Mumbai,Books,Truck,7,8.74,46.5,97.32,Normal,13.3,3.50,4.1,0,0.00,220.47,239.32,5,1,4.6


In [2]:
features = ["shipment_volume","weight_kg","distance_km","freight_cost",
            "inventory_level","state","category","transport_mode","weather"]
target = "delivery_days"

X = df[features]
y = df[target]

num_cols = ["shipment_volume","weight_kg","distance_km","freight_cost","inventory_level"]
cat_cols = ["state","category","transport_mode","weather"]

pre = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [3]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=180, max_depth=12, random_state=42, n_jobs=-1
    )
}

results = []

for name, estimator in models.items():
    pipe = Pipeline([("pre", pre), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

metrics = pd.DataFrame(results)
display(metrics.round(3))
metrics.to_csv("data/week4_model_metrics_from_notebook.csv", index=False)

,Model,MAE,RMSE,R2
0,Linear Regression,0.646,0.799,0.526
1,Random Forest,0.701,0.875,0.431


## Optimization concept

The next stage can formulate a Vehicle Routing Problem (VRP):

- Nodes = depot and delivery locations
- Objective = minimize distance/cost
- Constraints = vehicle capacity, route duration and delivery requirements
- Model output = candidate routes and resource allocation

A production implementation can use Google OR-Tools after obtaining reliable road-network travel times.